**SAPIANS** · SCC5819 · MÓDULO 04 — CONTRAFACTUAIS · MOLNAR, CAP. 15 · COMPANHEIRO TÉCNICO

# Contrafactuais por dentro — verificando as afirmações da aula

O walkthrough recortou o espaço de pares por legibilidade, leu a cerca
como uma contagem só e fixou o limiar em 0,5. Este caderno desfaz os
três atalhos e mede o que sobra.

[README do módulo](../README.md) · [a aula: `cf_walkthrough`](cf_walkthrough.ipynb) · [o modelo do curso](../../00-dataset/MODEL.md) · [outline da aula](../lecture/outline.md)

---

**O que este caderno verifica**

1. que o recorte dos 60 melhores movimentos não escondeu ótimo — o
   espaço completo de pares dá o mesmo melhor p (§1);
2. que a invalidez desta busca é baixa, e por qual cerca (§2);
3. que "curar" uma comorbidade passa na `gate_impossible` porque o
   portão do funil é **unidirecional no dado real** (§3);
4. que a conclusão do painel não depende do limiar 0,5 (§4).


In [1]:
%pip install -q xgboost scikit-learn matplotlib numpy pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
RANDOM_STATE = 42

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "README.md").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "README.md").exists():
    import tempfile
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))
import sapians as SP  # sem SP.aplicar(): este caderno não desenha, só imprime
import srag_60_model as M

g = M.load_gold(REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet")
xgb, _logit = M.fit_models(g)

# o paciente deste módulo, POR REGRA: contrafactual pede alguém com o que
# perder — p mais próximo de 0,8 no teste, empate por gold_id
teste = g[g["split"] == "test"].reset_index(drop=True)
p_teste = M.predict_proba(xgb, teste)
ordem = np.lexsort((teste["gold_id"].to_numpy(), np.abs(p_teste - 0.8)))
alto = teste.iloc[ordem[0]].copy()
alto["p_obito"] = float(p_teste[ordem[0]])



print(f"paciente de alto risco: gold_id {int(alto['gold_id'])}  "
      f"p(óbito) {SP.pt(alto['p_obito'], 4)}")
print(f"  idade {SP.pt(alto['idade_anos'], 0)} | "
      f"doses {SP.pt_int(alto['n_doses_antes_do_sintoma'])} | "
      f"meses desde mar/2020 {SP.pt(alto['meses_desde_mar2020'], 1)} | "
      f"fator de risco declarado: {bool(alto['fator_risc_portao'])}")
comorbs = [c for c in M.COMORBIDADES if str(alto[c]) == "sim"]
sintomas = [c for c in M.SINTOMAS if str(alto[c]) == "sim"]
print(f"  comorbidades sim: {comorbs}")
print(f"  sintomas sim: {sintomas}")

paciente de alto risco: gold_id 1266990  p(óbito) 0,7926
  idade 32 | doses 4 | meses desde mar/2020 46,1 | fator de risco declarado: True
  comorbidades sim: ['sind_down', 'neurologic', 'imunodepre', 'renal', 'out_morbi']
  sintomas sim: ['febre', 'tosse', 'dispneia', 'desc_resp', 'saturacao', 'dor_abd', 'fadiga']


## §0 — O mesmo espaço do walkthrough

As células abaixo reconstroem o espaço de movimentos e os lotes do
walkthrough (mesmas regras, mesmos números) para as medições em cima.

In [3]:
MOVES = {}
MOVES["idade_anos"] = [float(v) for v in range(0, 101, 10)]
MOVES["n_doses_antes_do_sintoma"] = [float(v) for v in range(0, 7)]
MOVES["meses_desde_mar2020"] = [float(v) for v in range(0, 59, 6)]
for c in M.BOOLEANAS:
    MOVES[c] = [False, True]
for c in M.CATEGORICAS:
    MOVES[c] = list(g[c].cat.categories)
# semana_epi fica fora: é atrelada a meses_desde_mar2020 — mover uma sem a
# outra fabricaria um calendário que não existe

# a que família cada uma das 40 features pertence — a tabela vive em
# tools/srag_60_model.py, uma só para os cinco módulos
GRUPO = dict(M.GRUPO_DE_FEATURE)


def candidatos_1(pac):
    out = []
    for c, vals in MOVES.items():
        for v in vals:
            if c in M.CATEGORICAS:
                if str(v) == str(pac[c]):
                    continue
            elif float(v) == float(pac[c]):
                continue
            out.append((c, v))
    return out


def aplicar(pac, lista_de_mudancas):
    """cada item é uma lista [(coluna, valor), ...]; devolve um frame."""
    linhas = []
    modelo_linha = {c: pac[c] for c in g.columns}
    for muds in lista_de_mudancas:
        linha = dict(modelo_linha)
        for c, v in muds:
            linha[c] = v
        linhas.append(linha)
    df = pd.DataFrame(linhas)
    return df.astype(g.dtypes)


c1 = candidatos_1(alto)
print(f"movimentos simples possíveis para este paciente: {SP.pt_int(len(c1))}")
n_grupos = pd.Series([GRUPO[c] for c, _ in c1]).value_counts()
print(n_grupos.to_string())


f1 = aplicar(alto, [[m] for m in c1])
p1 = M.predict_proba(xgb, f1)
delta = p1 - alto["p_obito"]
top60 = list(np.argsort(delta)[:60])
pares = [[c1[i], c1[j]] for a, i in enumerate(top60)
         for j in top60[a + 1:] if c1[i][0] != c1[j][0]]
f2 = aplicar(alto, pares)
p2 = M.predict_proba(xgb, f2)
ok2 = (p2 < 0.5) & ~M.gate_impossible(f2).to_numpy()

teste_alto = teste[M.predict_proba(xgb, teste) >= 0.5].reset_index(drop=True)
alto_risco = teste_alto


def candidatos_acionaveis(pac):
    d0 = int(pac["n_doses_antes_do_sintoma"])
    cand = [[("n_doses_antes_do_sintoma", float(d))] for d in range(d0 + 1, 7)]
    if not bool(pac["vacina_covid_declarada"]):
        cand += [[("vacina_covid_declarada", True)]]
        cand += [[("n_doses_antes_do_sintoma", float(d)),
                  ("vacina_covid_declarada", True)] for d in range(d0 + 1, 7)]
    return cand


blocos, indices = [], []
for k in range(len(alto_risco)):
    cand_k = candidatos_acionaveis(alto_risco.iloc[k])
    if cand_k:
        blocos.append(aplicar(alto_risco.iloc[k], cand_k))
        indices.extend([k] * len(cand_k))
lote = pd.concat(blocos, ignore_index=True)
p_lote = M.predict_proba(xgb, lote)
print(f"lotes reconstruídos: {SP.pt_int(len(f1))} simples, "
      f"{SP.pt_int(len(f2))} pares, {SP.pt_int(len(lote))} candidatos "
      f"acionáveis de {SP.pt_int(len(alto_risco))} pacientes")

movimentos simples possíveis para este paciente: 113
demografia     33
comorbidade    27
sintoma        26
tempo          10
contexto       10
dose            7


lotes reconstruídos: 113 simples, 1.691 pares, 3.195 candidatos acionáveis de 624 pacientes


## §1 — Exaustão sem recorte

O walkthrough limita os pares aos 60 melhores movimentos, por
legibilidade. Aqui o espaço completo — todos os pares de features
distintas — confirma que o recorte não escondeu ótimo nenhum.

In [4]:
todos_pares = [[c1[i], c1[j]] for i in range(len(c1))
               for j in range(i + 1, len(c1)) if c1[i][0] != c1[j][0]]
print(f"movimentos simples: {SP.pt_int(len(c1))}")
print(f"pares completos (features distintas): {SP.pt_int(len(todos_pares))}")
f_tudo = aplicar(alto, todos_pares)
p_tudo = M.predict_proba(xgb, f_tudo)
inv_tudo = M.gate_impossible(f_tudo).to_numpy()
ok_tudo = (p_tudo < 0.5) & ~inv_tudo
print(f"válidos que cruzam 0,5 no espaço COMPLETO: {SP.pt_int(ok_tudo.sum())}")
melhor_completo = float(p_tudo[ok_tudo].min())
print(f"melhor p válido no espaço completo: {SP.pt(melhor_completo, 4)}")
print("melhor p válido no recorte top-60 do walkthrough: "
      f"{SP.pt(float(p2[ok2].min()), 4)}")
print("=> o recorte do walkthrough é conveniência de leitura; o ótimo é o mesmo,")
print("   e a garantia vem da exaustão, não de sorte de semente.")

movimentos simples: 113
pares completos (features distintas): 6.129
válidos que cruzam 0,5 no espaço COMPLETO: 68
melhor p válido no espaço completo: 0,1995
melhor p válido no recorte top-60 do walkthrough: 0,1995
=> o recorte do walkthrough é conveniência de leitura; o ótimo é o mesmo,
   e a garantia vem da exaustão, não de sorte de semente.


## §2 — Invalidez por cerca

A mesma decomposição dos módulos 01–03, agora sobre candidatos de busca.

In [5]:
def decompor(frame):
    """as três cercas abertas, direto do módulo 00 — nada recalculado aqui"""
    m = M.gate_reasons(frame)
    return m["portao"], m["pre_campanha"], m["fora_coorte"]


for nome, frame in (("profundidade 1", f1), ("pares completos", f_tudo)):
    po, pr, fo = decompor(frame)
    print(f"{nome}: portão {SP.pct(po.mean())} | "
          f"pré-campanha {SP.pct(pr.mean())} | "
          f"fora da coorte {SP.pct(fo.mean())}")
print()
print("a invalidez aqui é BAIXA comparada ao LIME (30,6% lá) — a busca parte de")
print("um paciente real e muda 1–2 campos; o gerador do LIME sorteava os 40.")
print("métodos diferentes fabricam ficção em taxas diferentes, pela mesma régua.")

profundidade 1: portão 0,0% | pré-campanha 1,8% | fora da coorte 1,8%
pares completos: portão 0,0% | pré-campanha 3,3% | fora da coorte 3,6%

a invalidez aqui é BAIXA comparada ao LIME (30,6% lá) — a busca parte de
um paciente real e muda 1–2 campos; o gerador do LIME sorteava os 40.
métodos diferentes fabricam ficção em taxas diferentes, pela mesma régua.


## §3 — O portão é unidirecional

O achado que explica os contrafactuais de "cura": a cerca do funil barra
comorbidade sem fator declarado, não o contrário — porque o contrário
existe no dado real.

In [6]:
# o portão é unidirecional: comorbidade "sim" sem fator declarado contradiz
# o funil (0,00% em seis anos); o contrário — fator declarado sem nenhuma
# comorbidade nomeada — existe no dado real (o campo agregado veio marcado
# e o detalhamento não). Logo "curar" a imunodepressão do paciente passa
# na cerca:
cura = aplicar(alto, [[("imunodepre", "nao")]])
print(f"imunodepre -> nao, com fator_risc_portao=True herdado: "
      f"gate_impossible = {bool(M.gate_impossible(cura).iloc[0])}")
com_fator = g[g["fator_risc_portao"].astype(bool)]
comorb = M.COMORBIDADES
alguma = pd.concat([(com_fator[c].astype(str) == "sim") for c in comorb],
                   axis=1).any(axis=1)
print(f"no dado real: {SP.pct((~alguma).mean(), 2)} dos com fator declarado "
      f"não nomeiam comorbidade alguma — o estado 'declarado sem detalhe' existe")
n_cura = sum(1 for muds in todos_pares
             for c, v in muds if c in comorb and str(v) != "sim"
             and str(alto[c]) == "sim")
print(f"pares que exploram uma 'cura' de comorbidade deste paciente: {SP.pt_int(n_cura)}")
print("=> a cerca deixa passar de propósito; quem barra a cura é a lista de")
print("   alavancas — de novo: existir e estar ao alcance são perguntas distintas.")

imunodepre -> nao, com fator_risc_portao=True herdado: gate_impossible = False


no dado real: 0,54% dos com fator declarado não nomeiam comorbidade alguma — o estado 'declarado sem detalhe' existe
pares que exploram uma 'cura' de comorbidade deste paciente: 1.110
=> a cerca deixa passar de propósito; quem barra a cura é a lista de
   alavancas — de novo: existir e estar ao alcance são perguntas distintas.


## §4 — Sensibilidade do limiar

0,5 é convenção, não verdade clínica. A conta do painel, refeita.

In [7]:
for limiar in (0.5, 0.4, 0.3):
    flip_l = pd.Series(p_lote < limiar).groupby(np.asarray(indices)).any()
    n = int(flip_l.sum())
    print(f"limiar {SP.pt(limiar, 1)}: {SP.pt_int(n)}/{SP.pt_int(len(alto_risco))} "
          f"com contrafactual acionável ({SP.pct(n / len(alto_risco))})")
print()
print("o limiar é decisão de quem opera, não do método; a conta acompanha.")

limiar 0,5: 189/624 com contrafactual acionável (30,3%)
limiar 0,4: 58/624 com contrafactual acionável (9,3%)
limiar 0,3: 7/624 com contrafactual acionável (1,1%)

o limiar é decisão de quem opera, não do método; a conta acompanha.


## Fechamento — o que fica estabelecido

A busca exaustiva é o luxo dos espaços pequenos: cobertura completa,
determinismo, custo de milissegundos — e nenhuma das conclusões do
walkthrough depende de recorte, semente ou limiar único. O que o método
não resolve sozinho continua sendo o desenho: a lista de alavancas é uma
decisão sobre o mundo, não uma propriedade do modelo.

**Referências**

- Molnar, C. — *Interpretable Machine Learning*, 3ª ed., [cap. 15
  (Counterfactual Explanations)](https://christophm.github.io/interpretable-ml-book/counterfactual.html).
- Wachter, Mittelstadt & Russell (2018) — *Counterfactual Explanations
  without Opening the Black Box*, Harvard JOLT 31(2)
  ([arXiv:1711.00399](https://arxiv.org/abs/1711.00399)).
- Dandl, Molnar, Binder & Bischl (2020) — *Multi-Objective Counterfactual
  Explanations* ([arXiv:2004.11165](https://arxiv.org/abs/2004.11165)).
- As cercas, o funil e o modelo: módulo 00
  ([MODEL.md](../../00-dataset/MODEL.md),
  [gold/MANIFEST.md](../../00-dataset/gold/MANIFEST.md)).
- Stack pinado deste caderno: xgboost 3.4.1, scikit-learn 1.9.0,
  pandas 2.3.3, numpy 2.4.6 (`requirements.lock`).